# Noise robustness at N=9 (canonical pipeline)

**Purpose.** Test whether the single-soliton finding ("L-BFGS hurts under noisy
data") replicates in the multi-soliton regime.

**Why this is a rerun.** Previous attempt got 84% L² at σ=0 (should be ~0.97%).
Pipeline was broken. The σ=0 run here is the canary — if it doesn't reproduce
~0.97%, the notebook prints a warning and the noise comparison is invalid.

**Approx runtime:** ~2 h on Kaggle T4 (~30 min × 4 noise levels).

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

for p in ["/kaggle/input/kdv-core", "/kaggle/working", "."]:
    if (Path(p) / "kdv_core.py").exists():
        sys.path.insert(0, p)
        break
import kdv_core as K
print(f"device = {K.DEVICE}")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT.mkdir(parents=True, exist_ok=True)
print(f"output dir = {OUT}")


In [ ]:
K.quick_sanity_check(seed=99, adam_iter=500)

### Step 1 — Sweep σ ∈ {0%, 1%, 5%, 10%}

Same N=9 config as the scaling notebook (speeds 5.5→1.5, domain [-50, 82.5],
n_pde=44166, n_data=883). σ is expressed as a fraction of the tallest soliton's
peak amplitude. Both after-Adam and after-LBFGS L² are logged.

In [ ]:
SIGMAS = [0.00, 0.01, 0.05, 0.10]
N = 9
SEED = 99

cfg = K.make_config(N)
K.print_config(cfg)

results = []
for sigma in SIGMAS:
    tag = f"N{N}_noise_sig{int(sigma*100):02d}"
    ckpt_path = OUT / f"checkpoint_{tag}.pt"

    if ckpt_path.exists():
        print(f"\n[resume] {tag} already done, loading.")
        ck = K.load_checkpoint(ckpt_path)
        hist = ck["history"]
        results.append(dict(sigma_pct=int(sigma*100),
                            L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                            adam_time=hist["adam_time"], lbfgs_time=hist["lbfgs_time"]))
        continue

    print("\n" + "=" * 60)
    print(f"NOISE sigma={sigma*100:.0f}%  ({tag})")
    print("=" * 60)
    K.set_seed(SEED)
    batch = K.build_data(cfg, seed=SEED, noise_sigma=sigma)
    m = K.PINN(width=50).to(K.DEVICE)
    hist = K.train(m, batch, adam_iter=15000, lbfgs_iter=2000)
    K.save_checkpoint(ckpt_path, m, hist, cfg,
                      extras=dict(tag=tag, sigma=sigma, N=N, seed=SEED))
    print(f"  saved {ckpt_path}")

    results.append(dict(sigma_pct=int(sigma*100),
                        L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                        adam_time=hist["adam_time"], lbfgs_time=hist["lbfgs_time"]))
    pd.DataFrame(results).to_csv(OUT / "noise_results.csv", index=False)

df = pd.DataFrame(results)
df["lbfgs_helped"] = df["L2_final"] < df["L2_adam"]
print("\n--- Results ---")
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
df.to_csv(OUT / "noise_results.csv", index=False)


### Step 2 — Pipeline canary: σ=0 must reproduce ~0.97%

In [ ]:
sigma0_row = df[df["sigma_pct"] == 0].iloc[0]
l2_clean = sigma0_row["L2_final"]
print(f"sigma=0% Final L2 = {l2_clean:.4f}%")
print(f"Reference:          ~0.9754%")
if abs(l2_clean - 0.9754) < 0.5:
    print("[OK] Reproduces baseline -> pipeline healthy, noise results trustworthy.")
else:
    print("[WARN] Deviates significantly from scaling baseline.")
    print("       Run-to-run variance OR pipeline drift — inspect before trusting.")


### Step 3 — Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df["sigma_pct"], df["L2_adam"],  "o-", label="After Adam",   color="C0", lw=2, ms=8)
ax.plot(df["sigma_pct"], df["L2_final"], "s-", label="After L-BFGS", color="C3", lw=2, ms=8)
for _, row in df.iterrows():
    ax.annotate(f"{row['L2_final']:.2f}%", (row["sigma_pct"], row["L2_final"]),
                textcoords="offset points", xytext=(5, 8), fontsize=9)
ax.set_xlabel("noise sigma (% of peak amplitude)")
ax.set_ylabel("L2 error (%)")
ax.set_yscale("log")
ax.set_title(f"N={N} noise robustness — multi-soliton (canonical pipeline)")
ax.legend(); ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(OUT / "noise_robustness.png", dpi=140, bbox_inches="tight")
plt.show()


### Step 4 — Compare vs single-soliton paper

In [ ]:
SINGLE_PINN = {0: 0.07, 1: 0.83, 5: 5.51, 10: 12.2}
comparison = df.copy()
comparison["single_sol_PINN_pct"] = comparison["sigma_pct"].map(SINGLE_PINN)
comparison["degradation_factor"]  = (comparison["L2_final"] /
                                      df.loc[df["sigma_pct"]==0, "L2_final"].iloc[0])
print(comparison[["sigma_pct","L2_final","single_sol_PINN_pct","degradation_factor"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
comparison.to_csv(OUT / "noise_vs_single_soliton.csv", index=False)
print("\nDone. Outputs in", OUT)
